<font color='#007CE5'>**CLASIFICACIÓN DE NOTICIAS POR LOCALIZACIÓN CON REGEX**</font>

Archivo json obtenido de:
http://datos.gob.es/es/catalogo/a09002970-municipios-de-catalunya-geo

In [ ]:
import pandas as pd
import re
import json
from unidecode import unidecode
from langdetect import detect, DetectorFactory


# CONFIGURACIÓN 

DetectorFactory.seed = 0 

def norm_str(s):
    return unidecode(str(s).lower())

def fix_spaces(s):
    return re.sub(r"([a-z])([A-Z])", r"\1 \2", s)

def detectar_idioma_seguro(texto):
    if not isinstance(texto, str) or len(texto.split()) < 4:
        return "desconocido"
    try:
        lang = detect(texto)
        return lang if lang in ["es", "ca"] else "desconocido"
    except:
        return "desconocido"

def compile_terms_safe(terms, flexible=False):
    terms_norm = [unidecode(t.lower()) for t in terms]
    terms_sorted = sorted(set(terms_norm), key=len, reverse=True)
    if flexible:
        pat = "(" + "|".join(re.escape(t) for t in terms_sorted) + ")"
    else:
        pat = r"\b(" + "|".join(re.escape(t) for t in terms_sorted) + r")\b"
    return re.compile(pat, re.IGNORECASE)


# CARGAR MUNICIPIOS DESDE JSON

with open("comarques_municipis.json", "r", encoding="utf-8") as f:
    data = json.load(f)

cat_municipios = set()
for comarca, cities in data.items():
    for city in cities:
        name = city.get("city_name", "")
        if name:
            cat_municipios.add(name)


# DICCIONARIO MANUAL

cat_manual = {
    "catalunya","cataluña","catalonia","barcelona","girona","gerona","lleida","lerida","tarragona",
    "sabadell","terrassa","badalona","reus","l'hospitalet","hospitalet de llobregat",
    "sant andreu","tossa de mar","lloret de mar","martorell","poble-sec","horta-guinardo","la selva",
    "eixample", "l'eixample", "el prat", "barceloneta", "sant quinti de mediona", "perello",
    "mossos","mossos d'esquadra","policia de la generalitat", "palau-solita i plegamans", "castellbisbal",
    "sentmenat", "cornella", "camarles", "vilanova i la geltru", "castelldefels", "montcada", "teia", "salt",
    "girones", "fonollosa", "igualada", "vilanova del cami", "santa coloma de farners", "mataro"
    "barbera del valles", "poblenou", "valls", "montblanc", "sant adria de besos", "gava", "costa brava",
    "el pont de vilomara", "rocafort", "sant joan", "ripollet", "cornella de llobregat", "TSJC", "olot",
    "baix llobregat", "valles occidental", "barcelones", "rosa peral", "la merce", "independentista", "independentisme",
    "artos", "helena jubany", "sagrada familia", "baix emporda", "palafrugell", "calella", "calella de palafrugell",
    "santa coloma", "cerdanyola", "valles", "tunel de vallvidera", "tribunal superior de justicia de catalunya",
    "aeroport del prat", "vendrell", "la mina", "mercat de santa caterina", "abrera", "generalitat", "gracia", "sant adria",
    "vila olimpica", "esparreguera", "nou barris", "el prat de llobregat", "premia", "premia de mar", "l'emporda", "maresme",
    "igualada"
}


out_terms = {
"aguadulce","aguilas","albacete","alcala de guadaira","alcala de henares","alcantarilla","alcazar de san juan",
"alcorcon","algeciras","alicante","aljaraque","almeria","almonte","almuñecar","alora","altea","aranda de duero",
"arcos de la frontera","arrecife","avila","aviles","ayamonte","badajoz","baena","barbate","barakaldo","basauri", 
"benalmadena","benavente","benicarlo","benidorm","benissa","bilbao","boadilla del monte","boiro","burjassot",
"burgos","caceres","cadiz","calahorra","calpe","camas","camargo","cangas","caravaca de la cruz","carmona",
"carrion de los cespedes","cartagena","castellon","castro urdiales","ceuta","chiclana de la frontera","ciudad real",
"ciudad rodrigo","collado villalba","colmenar viejo","cordoba","coslada","cuenca","don benito","donostia",
"dos hermanas","ecija","el ejido","el puerto de santa maria","el prat de llobregat","elche","estepona","ferrol",
"fuengirola","fuenlabrada","gandia","getafe","gijon","granada","guadalajara","guadix","hellin","huelva","huesca",
"ibiza","illescas","irun","isla cristina","jaen","jerez de la frontera","jinamar","la coruña","la laguna",
"la linea de la concepcion","la orotava","las palmas de gran canaria","leon","linares","logroño","lorca","los barrios",
"los realejos","lugo","madrid","majadahonda","malaga","manacor","marbella","marin","martos","melilla","merida",
"mieres","miranda de ebro","mislata","molina de segura","monforte de lemos","monzon","mostoles","motril","murcia",
"navalmoral de la mata","navia","nerja","ourense","oviedo","palencia","palma de mallorca","pamplona","paterna",
"peniscola","pinto","plasencia","ponferrada","pontevedra","puertollano","puerto del rosario","puerto real",
"puerto de la cruz","puerto de sagunto","redondela","rincon de la victoria","roquetas de mar","salamanca",
"san fernando","san javier","san sebastian","san vicente del raspeig","santander","santa cruz de tenerife",
"santa lucia de tirajana","santa pola","sanlucar de barrameda","sestao","segovia","sevilla","soria","telde",
"teruel","toledo","torrelavega","torremolinos","torrevieja","torrijos","ubeda","utrera","valdepeñas","valencia",
"valladolid","vigo","vilagarcia de arousa","villajoyosa","villanueva de la serena","villena","vitoria","zamora",
"zaragoza","doncaster","tailandia","republica checa","marroc", "daniel sancho", "ana julia", "paris","londres","nova york",
"nueva york","berlin","roma","tailandia", "castelló", "valència", "audiència de madrid", "torrejon de ardoz", "comunitat de madrid",
"washington", "estats units", "fbi", "comandancia de madrid", "cabanes", "belgica"
}


# PATRONES REGEX

PAT_CAT_NORMAL = compile_terms_safe(cat_municipios.union(cat_manual))
PAT_CAT_CRITICOS = compile_terms_safe(cat_manual, flexible=True)  
PAT_OUT = compile_terms_safe(out_terms)


def rule_locate(text):
    t = fix_spaces(text)
    t = norm_str(t)

    if re.search(r"\bmadrid\b", t): 
        return "España/Extranjero"
    if re.search(r"\bvalencia\b", t): 
        return "España/Extranjero"

    cat_hits = len(PAT_CAT_NORMAL.findall(t)) + len(PAT_CAT_CRITICOS.findall(t))
    out_hits = len(PAT_OUT.findall(t))


    if cat_hits >= 2 * out_hits and cat_hits > 0:
        return "Cataluña"
    elif out_hits > 0:
        return "España/Extranjero"
    else:
        return "España/Extranjero"

df = pd.read_csv("noticias_unidas.csv", sep=";")

if "text" not in df.columns and {"Título","Primeros_Párrafos"}.issubset(df.columns):
    df["text"] = df["Título"].astype(str).str.strip() + " " + df["Primeros_Párrafos"].astype(str).str.strip()


df["idioma"] = df["text"].apply(detectar_idioma_seguro)


df["loc_final"] = df["text"].apply(rule_locate)


df.to_csv("resultados_localizacion.csv", sep=";", index=False, encoding="utf-8-sig")
print("CSV final guardado: resultados_localizacion.csv")


print(df["loc_final"].value_counts())